<a href="https://colab.research.google.com/github/Mahmoud-Altras/Riyadh_Fleet_WFM_Audit/blob/main/Delivery_SLA_Financial_Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

print(" Starting Data Generation for Last-Mile Delivery WFM...")

# 1. Generate Dim_Rider (200 Riders)
np.random.seed(42)
rider_ids = [f"R-{str(i).zfill(3)}" for i in range(1, 201)]
hubs = ['North Riyadh', 'South Riyadh', 'East Riyadh', 'West Riyadh']
vehicles = ['Car', 'Motorcycle']

dim_rider = pd.DataFrame({
    'Rider_ID': rider_ids,
    'Hub': np.random.choice(hubs, 200, p=[0.3, 0.2, 0.25, 0.25]),
    'Vehicle_Type': np.random.choice(vehicles, 200, p=[0.4, 0.6]),
    'Cost_Per_Hour': np.random.choice([25, 30, 35], 200, p=[0.5, 0.3, 0.2])
})
dim_rider.to_csv('Dim_Rider.csv', index=False)
print(" Dim_Rider.csv created!")

# 2. Generate Fact_Roster (1 Month of Scheduled Shifts - June 2026)
dates = pd.date_range(start='2026-06-01', end='2026-06-30')
roster_data = []

for date in dates:
    # 80% of riders are scheduled each day
    scheduled_riders = random.sample(rider_ids, int(200 * 0.8))
    for rider in scheduled_riders:
        # Standard shift: 10:00 AM to 08:00 PM
        shift_start = datetime.combine(date, datetime.strptime("10:00:00", "%H:%M:%S").time())
        shift_end = shift_start + timedelta(hours=10)
        roster_data.append([date.date(), rider, shift_start, shift_end])

fact_roster = pd.DataFrame(roster_data, columns=['Date', 'Rider_ID', 'Scheduled_Start', 'Scheduled_End'])
fact_roster.to_csv('Fact_Roster.csv', index=False)
print(" Fact_Roster.csv created!")

# 3. Generate Fact_Rider_Logs (Intraday App Status - The SLA Trap)
log_data = []
statuses = ['Available', 'Delivering', 'Break', 'Vehicle Issue']

for index, row in fact_roster.iterrows():
    current_time = row['Scheduled_Start']
    end_time = row['Scheduled_End']
    rider = row['Rider_ID']
    hub = dim_rider[dim_rider['Rider_ID'] == rider]['Hub'].values[0]

    while current_time < end_time:
        #  THE TRAP: North Riyadh riders fake "Vehicle Issues" or "Break" at 1:00 PM peak!
        if hub == 'North Riyadh' and 13 <= current_time.hour < 15:
            status = np.random.choice(['Vehicle Issue', 'Break'], p=[0.7, 0.3])
            duration_mins = random.randint(30, 60)
        else:
            # Normal behavior
            status = np.random.choice(statuses, p=[0.3, 0.6, 0.05, 0.05])
            duration_mins = random.randint(15, 45)

        log_data.append([rider, current_time, status, duration_mins])
        current_time += timedelta(minutes=duration_mins)

fact_logs = pd.DataFrame(log_data, columns=['Rider_ID', 'Timestamp', 'App_Status', 'Duration_Minutes'])
fact_logs.to_csv('Fact_Rider_Logs.csv', index=False)
print(" Fact_Rider_Logs.csv created! (With SLA anomalies planted for North Riyadh)")

# Code to auto-download in Google Colab
try:
    from google.colab import files
    files.download('Dim_Rider.csv')
    files.download('Fact_Roster.csv')
    files.download('Fact_Rider_Logs.csv')
    print("📥 Downloading files to your machine...")
except:
    print("Files saved to current local directory.")

🚀 Starting Data Generation for Last-Mile Delivery WFM...
✅ Dim_Rider.csv created!
✅ Fact_Roster.csv created!
✅ Fact_Rider_Logs.csv created! (With SLA anomalies planted for North Riyadh)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading files to your machine...
